In [1]:
import numpy as np
from cobra.util.array import create_stoichiometric_matrix
from docplex.mp.model import Model
from cobra.io import read_sbml_model

In [2]:
"""
Simple Test Implementation: MILP for Shortest Elementary Modes
Based on von Kamp & Klamt (2014) - Equations 1, 2, 7-11

This is a minimal working example to test the core MILP formulation.
"""

import numpy as np
from docplex.mp.model import Model

def enumerate_k_shortest_ems(N, k=5, reversible_pairs=None):
    """
    Enumerate k shortest EMs using exclusion constraints (Equation 11).
    
    Args:
        N: Stoichiometric matrix
        irreversible_reactions: List of irreversible reaction indices
        k: Number of shortest EMs to find
        reversible_pairs: List of (forward_idx, backward_idx) tuples
        
    Returns:
        List of EM dictionaries
    """
    m, n = N.shape
    ems = []
    
    # Create base model
    mdl = Model(name="k_Shortest_EMs")
    
    # Variables
    r = mdl.continuous_var_list(n, lb=0, name='r')
    z = mdl.binary_var_list(n, name='z')
    
    # Steady state constraints
    for i in range(m):
        mdl.add_constraint(
            mdl.sum(N[i, j] * r[j] for j in range(n)) == 0,
            ctname=f'steady_state_{i}'
        )
    
    # Indicator constraints
    for i in range(n):
        mdl.add_indicator(z[i], r[i] == 0, active_value=0, name=f'ind_zero_{i}')
        mdl.add_indicator(z[i], r[i] >= 1, active_value=1, name=f'ind_active_{i}')
    
    # Zero-flux constraint
    if reversible_pairs:
        for fwd, bwd in reversible_pairs:
            mdl.add_constraint(z[fwd] + z[bwd] <= 1, ctname=f'rev_pair_{fwd}_{bwd}')
    
    # At least one reaction active
    mdl.add_constraint(mdl.sum(z) >= 1, ctname='at_least_one')
    
    # Objective
    mdl.minimize(mdl.sum(z))
    
    # Iteratively find k shortest EMs
    for iteration in range(k):
        solution = mdl.solve(log_output=False)
        
        if solution is None:
            print(f"No more EMs found after {iteration} iterations")
            break
        
        # Extract solution
        r_values = [r[i].solution_value for i in range(n)]
        z_values = [int(z[i].solution_value) for i in range(n)]
        
        em = {
            'fluxes': r_values,
            'active_reactions': [i for i, val in enumerate(z_values) if val == 1],
            'size': sum(z_values)
        }
        ems.append(em)
        
        # Add exclusion constraint - Equation (11)
        # sum(z_tilde[i] * z[i]) <= sum(z_tilde[i]) - 1
        mdl.add_constraint(
            mdl.sum(z_values[i] * z[i] for i in range(n)) <= sum(z_values) - 1,
            ctname=f'exclusion_{iteration}'
        )
        
        print(f"EM {iteration+1}: size={em['size']}, reactions={em['active_reactions']}")
    
    return ems    

In [3]:
model_name = '../../models/M_model'
model = read_sbml_model(model_name + ".xml")
S = create_stoichiometric_matrix(model)

n_original = len(model.reactions)
fwd = [index for index, reaction in enumerate(model.reactions) if reaction.reversibility]

rev_pairs = []
for rev_off, i in enumerate(fwd):
    bwd_idx = n_original + rev_off
    S = np.append(S, np.transpose([-S[:, i]]), axis=1)
    rev_pairs.append((i, bwd_idx))



efms_M = enumerate_k_shortest_ems(S, k=5020, reversible_pairs=rev_pairs)

EM 1: size=2, reactions=[0, 1]
EM 2: size=3, reactions=[3, 4, 5]
EM 3: size=3, reactions=[1, 2, 4]
EM 4: size=3, reactions=[1, 3, 4]
EM 5: size=3, reactions=[2, 4, 5]
No more EMs found after 5 iterations


In [4]:
model_name = '../../models/PQS_model'
model = read_sbml_model(model_name + ".xml")
S = create_stoichiometric_matrix(model)

n_original = len(model.reactions)
fwd = [index for index, reaction in enumerate(model.reactions) if reaction.reversibility]

rev_pairs = []
for rev_off, i in enumerate(fwd):
    bwd_idx = n_original + rev_off
    S = np.append(S, np.transpose([-S[:, i]]), axis=1)
    rev_pairs.append((i, bwd_idx))



efms_PQS = enumerate_k_shortest_ems(S, k=5020, reversible_pairs=rev_pairs)

EM 1: size=3, reactions=[0, 1, 7]
EM 2: size=3, reactions=[6, 9, 11]
EM 3: size=3, reactions=[0, 2, 8]
EM 4: size=4, reactions=[0, 3, 4, 10]
EM 5: size=5, reactions=[0, 2, 3, 5, 9]
EM 6: size=5, reactions=[0, 3, 4, 6, 9]
No more EMs found after 6 iterations


In [5]:
model_name = '../../models/ecoli5010_no_b'
model = read_sbml_model(model_name + ".xml")
S = create_stoichiometric_matrix(model)

n_original = len(model.reactions)
fwd = [index for index, reaction in enumerate(model.reactions) if reaction.reversibility]

rev_pairs = []
for rev_off, i in enumerate(fwd):
    bwd_idx = n_original + rev_off
    S = np.append(S, np.transpose([-S[:, i]]), axis=1)
    rev_pairs.append((i, bwd_idx))



efms_ecoli = enumerate_k_shortest_ems(S, k=5020, reversible_pairs=rev_pairs)

EM 1: size=2, reactions=[26, 29]
EM 2: size=15, reactions=[5, 6, 7, 8, 12, 13, 37, 44, 45, 46, 48, 50, 55, 63, 64]
EM 3: size=15, reactions=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 37, 46, 55, 63, 64]
EM 4: size=15, reactions=[0, 1, 3, 4, 5, 6, 7, 8, 9, 37, 46, 48, 55, 63, 64]
EM 5: size=16, reactions=[1, 2, 5, 6, 7, 8, 12, 13, 37, 44, 45, 46, 50, 55, 63, 64]
EM 6: size=16, reactions=[0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 37, 46, 51, 55, 63, 64]
EM 7: size=16, reactions=[5, 6, 7, 8, 12, 13, 32, 34, 37, 44, 45, 46, 50, 55, 63, 64]
EM 8: size=16, reactions=[0, 1, 3, 4, 5, 6, 7, 8, 9, 32, 34, 37, 46, 55, 63, 64]
EM 9: size=17, reactions=[0, 1, 3, 4, 5, 6, 7, 8, 32, 33, 37, 46, 48, 55, 63, 64, 76]
EM 10: size=17, reactions=[0, 1, 3, 4, 5, 6, 7, 8, 32, 33, 34, 37, 46, 55, 63, 64, 76]
EM 11: size=17, reactions=[0, 1, 2, 3, 4, 5, 6, 7, 8, 32, 33, 37, 46, 55, 63, 64, 76]
EM 12: size=17, reactions=[5, 6, 7, 8, 9, 10, 12, 13, 37, 44, 45, 46, 50, 51, 55, 63, 64]
EM 13: size=18, reactions=[5, 6, 7, 8, 12, 13, 40, 